In [47]:
# --- CELL 1: ENVIRONMENT & INGESTION ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Set visual style for EDA
sns.set_theme(style="whitegrid")

print("Step 1: Loading raw Pipeline ILI datasets...")

# Load the raw CSV files
df_2015 = pd.read_csv('V-T ILI 2015 - Copy - Sheet1.csv', low_memory=False)
df_2023 = pd.read_csv('V-T_ILI_2023_Cleaned.csv', low_memory=False)

# Clean up column names by stripping accidental whitespace
df_2015.columns = df_2015.columns.str.strip()
df_2023.columns = df_2023.columns.str.strip()

print(f"✅ 2015 Data Loaded: {len(df_2015):,} rows found.")
print(f"✅ 2023 Data Loaded: {len(df_2023):,} rows found.")

Step 1: Loading raw Pipeline ILI datasets...
✅ 2015 Data Loaded: 14,691 rows found.
✅ 2023 Data Loaded: 31,421 rows found.


In [48]:
# ==========================================
# ZEROING THE ODOMETER (Normalizing Distances)
# ==========================================

print("--- Zeroing 2015 Dataset ---")
# 1. Grab the exact distance of the very first row (the ~56m valve)
start_dist_15 = df_2015['Pigrun log distance (m)'].iloc[0]
print(f"Original 2015 start distance: {start_dist_15} m")

# 2. Subtract this value from the entire column at once
df_2015['Pigrun log distance (m)'] = df_2015['Pigrun log distance (m)'] - start_dist_15


print("--- Zeroing 2023 Dataset ---")
# 1. Grab the exact distance of the very first row (the ~56m valve)
start_dist_23 = df_2023['Pigrun log distance (m)'].iloc[0]
print(f"Original 2023 start distance: {start_dist_23} m")

# 2. Subtract this value from the entire column at once
df_2023['Pigrun log distance (m)'] = df_2023['Pigrun log distance (m)'] - start_dist_23


--- Zeroing 2015 Dataset ---
Original 2015 start distance: 57.415 m
--- Zeroing 2023 Dataset ---
Original 2023 start distance: 56.431 m


In [49]:
# Columns to keep
cols_to_keep = [
    'Length (mm)',
    'Width (mm)',
    'Depth (mm)',
    'Up weld dist (m)',
    'Clock (hh:mm)',
    'Pigrun log distance (m)',
    'Feature type',
    'Feature identification'
]

# Keep only these columns
df_2023 = df_2023[cols_to_keep]

In [50]:
# In Feature type column keep only Anomaly and in Feature Identification column keep only Metal loss
df_Anomaly23 = df_2023[df_2023['Feature type'].isin(['Anomaly']) & df_2023['Feature identification'].isin(['Metal loss'])]

In [51]:
# Columns to keep
cols_to_keep = [
    'Length (mm)',
    'Width (mm)',
    'Depth (mm)',
    'Up weld dist (m)',
    'Clock (hh:mm)',
    'Pigrun log distance (m)',
    'Feature type',
    'Feature identification',
    
]

# Keep only these columns
df_2015 = df_2015[cols_to_keep]


In [52]:
# In Feature type column keep only Anomaly and in Feature Identification column keep only Metal loss
df_Anomaly15 = df_2015[df_2015['Feature type'].isin(['Anomaly']) & df_2015['Feature identification'].isin(['Anomaly'])]

In [53]:
df_Anomaly15.shape

(8898, 8)

In [54]:
df_Anomaly23.shape

(25064, 8)

In [55]:
## Helper function to convert "HH:MM" strings into a 360-degree mathematical angle
def clock_to_degrees(clock_str):
    
    parts = str(clock_str).split(':')
    hours = float(parts[0])
    minutes = float(parts[1])
    # 1 hour = 30 degrees, 1 minute = 0.5 degrees
    degrees = (hours * 30.0) + (minutes * 0.5)
    return degrees 

# Create safe copies of the data and apply the mathematical conversion
df_Anomaly15 = df_Anomaly15.copy()
df_Anomaly23 = df_Anomaly23.copy()

df_Anomaly15['Angle_Deg'] = df_Anomaly15['Clock (hh:mm)'].apply(clock_to_degrees)
df_Anomaly23['Angle_Deg'] = df_Anomaly23['Clock (hh:mm)'].apply(clock_to_degrees)

In [56]:
# ==========================================
# 2. FILTER FOR THE SAFE ZONE
# ==========================================
print("Filtering for anomalies in the aligned zone...\n")

safe_anomalies_15 = df_Anomaly15[df_Anomaly15['Pigrun log distance (m)'] <= 7315]
safe_anomalies_23 = df_Anomaly23[df_Anomaly23['Pigrun log distance (m)'] <= 7340]

Filtering for anomalies in the aligned zone...



In [57]:
safe_anomalies_23 = safe_anomalies_23.reset_index()
safe_anomalies_15 = safe_anomalies_15.reset_index()

In [58]:
safe_anomalies_15

,index,Length (mm),Width (mm),Depth (mm),Up weld dist (m),Clock (hh:mm),Pigrun log distance (m),Feature type,Feature identification,Angle_Deg
0,4,28.0,25.0,0.857,5.058,01:00,10.251,Anomaly,Anomaly,30.0
1,6,41.0,41.0,0.857,3.414,11:40,14.152,Anomaly,Anomaly,350.0
2,7,40.0,36.0,0.714,5.367,00:20,16.105,Anomaly,Anomaly,10.0
3,8,48.0,69.0,1.142,5.398,11:30,16.136,Anomaly,Anomaly,345.0
4,10,23.0,28.0,0.785,6.004,10:30,28.770,Anomaly,Anomaly,315.0
...,...,...,...,...,...,...,...,...,...,...
1769,2690,30.0,23.0,0.857,5.301,06:20,7307.338,Anomaly,Anomaly,190.0
1770,2691,23.0,28.0,0.857,10.398,02:30,7312.435,Anomaly,Anomaly,75.0
1771,2692,15.0,23.0,0.857,10.945,02:25,7312.982,Anomaly,Anomaly,72.5
1772,2693,0.0,0.0,0.000,11.195,10:20,7313.232,Anomaly,Anomaly,310.0


In [59]:
print("Step 1: Extracting Welds to build the Spool Map...")

# 1. Get the Weld Distances for the Safe Zone (0 - 7.3km)
# We pull this from the raw dataframes you loaded in Cell 1
welds_15 = df_2015[(df_2015['Feature type'].str.contains('Weld', na=False)) & (df_2015['Pigrun log distance (m)'] <= 7315)]
welds_23 = df_2023[(df_2023['Feature type'].str.contains('Weld', na=False)) & (df_2023['Pigrun log distance (m)'] <= 7340)]

# Create arrays of the physical weld locations
weld_dists_15 = welds_15['Pigrun log distance (m)'].sort_values().values
weld_dists_23 = welds_23['Pigrun log distance (m)'].sort_values().values

print(f"Found {len(weld_dists_15)} Welds in 2015 (Safe Zone)")
print(f"Found {len(weld_dists_23)} Welds in 2023 (Safe Zone)")

Step 1: Extracting Welds to build the Spool Map...
Found 679 Welds in 2015 (Safe Zone)
Found 679 Welds in 2023 (Safe Zone)


In [60]:
# ==========================================
# Step 2: Assign a "Spool ID" to every anomaly
# ==========================================
print("\nStep 2: Assigning Spool IDs to all anomalies...")

# np.searchsorted finds which two welds an anomaly sits between.
# E.g., if Welds are at 0m, 12m, 24m, an anomaly at 15m gets assigned to Spool #2.
safe_anomalies_15 = safe_anomalies_15.copy()
safe_anomalies_23 = safe_anomalies_23.copy()

safe_anomalies_15['Spool_ID'] = np.searchsorted(weld_dists_15, safe_anomalies_15['Pigrun log distance (m)'])
safe_anomalies_23['Spool_ID'] = np.searchsorted(weld_dists_23, safe_anomalies_23['Pigrun log distance (m)'])


Step 2: Assigning Spool IDs to all anomalies...


In [62]:
# ==========================================
# Step 3: SPOOL-BY-SPOOL MATCHING ALGORITHM
# ==========================================
print("Step 3: Running Spool-by-Spool Matching Algorithm...")

matched_results = []

# Extremely strict tolerances because we are locked inside a specific pipe!
MAX_UPWELD = 0.05  # 5 cm from the upstream weld
MAX_ANGLE = 20.0   # 20 degrees 

# Loop through the 2015 anomalies. iterrows takes one row of the safe_anomalies_15
# it saves the row's index as index and all other data as a15.
for index, a15 in safe_anomalies_15.iterrows():
    
    spool_id_15 = a15['Spool_ID']
    upweld_15 = a15['Up weld dist (m)']
    angle_15 = a15['Angle_Deg']
    
    # ----------------------------------------------------
    # FILTER 1: Isolate to the EXACT SAME SPOOL
    
    # Here only those rows of safe_anomalies_23 are kept in candidate dataframe who's spool_id match with 
    # the spool_id_15 
    # ----------------------------------------------------
    candidates = safe_anomalies_23[safe_anomalies_23['Spool_ID'] == spool_id_15].copy()
    
    # ----------------------------------------------------
    # FILTER 2: Up-Weld Pinpoint
    
    # We check the Up weld distance difference condition next.
    # If it fits within the tolerance then it can be considered in candidate.
    # ----------------------------------------------------
    if not candidates.empty:
        upweld_diff = np.abs(candidates['Up weld dist (m)'] - upweld_15)
        candidates = candidates[upweld_diff <= MAX_UPWELD]
        
    # ----------------------------------------------------
    # FILTER 3: Clock Angle
    
    # Next we check the Clock Angle difference condition next.
    # If it fits within the tolerance then it can be considered in candidate.
    # ----------------------------------------------------
    if not candidates.empty:
        angle_diff = np.abs(candidates['Angle_Deg'] - angle_15)
        angle_match = (angle_diff <= MAX_ANGLE) | (angle_diff >= 360 - MAX_ANGLE)
        candidates = candidates[angle_match]
        
    # ----------------------------------------------------
    # TIE-BREAKER: Closest local pit

    # This is last condition. 
    # In case more than one anomaly of 2023 data is satisfying the conditions for matching with 2015 anomaly
    # then we find that which anomaly in 2023 is closer to the 2015 anomaly.  
    # ----------------------------------------------------
    if not candidates.empty:
        candidates['UpWeld_Error'] = np.abs(candidates['Up weld dist (m)'] - upweld_15)
        best_match = candidates.sort_values(by='UpWeld_Error').iloc[0]
        
        matched_results.append({
            'Spool ID': spool_id_15,
            'Dist 2015 (m)': a15['Pigrun log distance (m)'],
            'Dist 2023 (m)': best_match['Pigrun log distance (m)'],
            'UpWeld 2015 (m)': upweld_15,
            'UpWeld 2023 (m)': best_match['Up weld dist (m)'],
            'Clock 2015': a15['Clock (hh:mm)'],
            'Clock 2023': best_match['Clock (hh:mm)'],
            'Depth 2015 (mm)': a15['Depth (mm)'],
            'Depth 2023 (mm)': best_match['Depth (mm)']
        })

df_spool_matches = pd.DataFrame(matched_results)
print(f"\n✅ SUCCESS! Matched {len(df_spool_matches):,} anomalies using Joint-to-Joint mapping.")
display(df_spool_matches.head(15))

Step 3: Running Spool-by-Spool Matching Algorithm...

✅ SUCCESS! Matched 688 anomalies using Joint-to-Joint mapping.


,Spool ID,Dist 2015 (m),Dist 2023 (m),UpWeld 2015 (m),UpWeld 2023 (m),Clock 2015,Clock 2023,Depth 2015 (mm),Depth 2023 (mm)
0,10,82.748,84.758,2.648,2.691,05:05,04:26,1.142,0.428
1,11,92.322,94.589,0.600,0.607,05:50,06:04,0.785,0.500
2,12,112.800,115.324,9.027,8.998,07:30,07:02,0.785,0.643
3,12,115.075,117.649,11.302,11.323,05:40,05:22,1.142,1.499
4,12,115.109,117.649,11.336,11.323,05:10,05:22,1.357,1.499
5,15,133.274,136.055,1.102,1.114,05:30,05:42,1.071,0.500
6,15,133.607,136.373,1.435,1.432,05:40,05:54,0.785,0.500
7,16,145.834,148.241,5.839,5.874,06:10,06:12,0.785,NaN
8,16,145.901,148.241,5.906,5.874,06:00,06:12,1.071,NaN
9,17,152.010,154.473,5.709,5.719,04:50,05:04,0.857,NaN
